# Chapter 7 — Search In-Depth
## Topic 4: Hybrid Search — BM25 + Dense via Reciprocal Rank Fusion (RRF)

**Run cells in order. Each cell is a self-contained module.**

- Module 1: Setup — shared knowledge base, BM25 index, dense model
- Module 2: Run BM25 and Dense independently — see the two ranked lists side by side
- Module 3: Reciprocal Rank Fusion implemented from scratch
- Module 4: Hybrid vs BM25-only vs Dense-only — head-to-head on Hinglish + English queries
- Module 5: k-value sensitivity — how the RRF constant changes the fused ranking

**Install:** `pip install rank-bm25 sentence-transformers numpy qdrant-client`


## Module 1: Setup — Knowledge Base, BM25 Index, Dense Model

Run this first. Builds everything Modules 2-5 depend on.

In [1]:
"""
MODULE 1: Setup

Builds:
  - KNOWLEDGE_BASE   : 6 chunks from 4 FD source documents
  - bm25             : BM25Okapi index over the knowledge base
  - dense_model      : paraphrase-multilingual-MiniLM-L12-v2
  - kb_embeddings    : pre-computed dense embeddings for the knowledge base
  - tokenize()       : shared tokenizer
"""

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------------------------
# Knowledge base: same 6 chunks used across Topics 1, 2, 3 for continuity
# -----------------------------------------------------------------------
KNOWLEDGE_BASE = [
    {"id": 0, "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf", "doc_type": "faq"},
    {"id": 1, "text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.", "source": "04_FD_Policy_Reference.pdf", "doc_type": "policy"},
    {"id": 2, "text": "This SOP covers Fixed Deposit premature withdrawal. Step 1: Verify FD account eligibility. Step 2: Calculate penalty. Step 3: Process premature withdrawal and apply 1 percent penalty. Step 4: Update FD records.", "source": "03_FD_SOPs.pdf", "doc_type": "sop"},
    {"id": 3, "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.", "source": "02_FD_Product_Guide.pdf", "doc_type": "product"},
    {"id": 4, "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "source": "02_FD_Product_Guide.pdf", "doc_type": "product"},
    {"id": 5, "text": "Early exit from your deposit account will attract a deduction from your returns.", "source": "04_FD_Policy_Reference.pdf", "doc_type": "policy"},
]

CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]


def tokenize(text: str) -> list:
    """Lowercase whitespace tokenizer — same convention as Topic 1."""
    return text.lower().split()


# -----------------------------------------------------------------------
# BM25 index (Topic 1 pattern)
# -----------------------------------------------------------------------
tokenized_corpus = [tokenize(t) for t in CORPUS_TEXTS]
bm25 = BM25Okapi(tokenized_corpus, k1=1.2, b=0.75)
print("BM25 index built.")

# -----------------------------------------------------------------------
# Dense model + pre-embedded knowledge base (Topic 2 pattern)
# -----------------------------------------------------------------------
print("Loading dense embedding model (may take a moment on first run)...")
dense_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kb_embeddings = dense_model.encode(CORPUS_TEXTS, normalize_embeddings=True, show_progress_bar=False)
print(f"Dense model loaded. Embeddings shape: {kb_embeddings.shape}")

print("\nKnowledge base:")
for doc in KNOWLEDGE_BASE:
    print(f"  Doc {doc['id']} [{doc['doc_type'].upper():<7}]: {doc['text'][:65]}...")

print("\nModule 1 complete. Run Module 2.")



BM25 index built.
Loading dense embedding model (may take a moment on first run)...
Dense model loaded. Embeddings shape: (6, 384)

Knowledge base:
  Doc 0 [FAQ    ]: Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Doc 1 [POLICY ]: Fixed Deposit premature closure is allowed subject to applicable ...
  Doc 2 [SOP    ]: This SOP covers Fixed Deposit premature withdrawal. Step 1: Verif...
  Doc 3 [PRODUCT]: The Fixed Deposit interest rate for 24 months is 7.1 percent per ...
  Doc 4 [PRODUCT]: Senior citizens receive an additional 0.5 percent interest on all...
  Doc 5 [POLICY ]: Early exit from your deposit account will attract a deduction fro...

Module 1 complete. Run Module 2.


## Module 2: Run BM25 and Dense Independently

Produces two separate ranked lists for the same query — the raw inputs to RRF.
Shows the score-scale mismatch that motivates rank-based fusion instead of score averaging.

**Requires:** Module 1


In [2]:
"""
MODULE 2: Independent BM25 and Dense Retrieval

Returns each retriever's ranked list as [(rank, doc_id, score), ...]
so Module 3 can consume them directly for RRF.
"""

def bm25_ranked_list(query: str, top_k: int = 6) -> list:
    """Returns [(rank, doc_id, bm25_score), ...] sorted by BM25 score descending.
    Only documents with score > 0 are included (BM25 convention: 0 = no overlap)."""
    scores = bm25.get_scores(tokenize(query))
    scored = [(KNOWLEDGE_BASE[i]["id"], scores[i]) for i in range(len(scores)) if scores[i] > 0]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [(rank + 1, doc_id, score) for rank, (doc_id, score) in enumerate(scored[:top_k])]


def dense_ranked_list(query: str, top_k: int = 6) -> list:
    """Returns [(rank, doc_id, cosine_score), ...] sorted by cosine similarity descending.
    Dense retrieval always returns a score for every document (no hard zero cutoff)."""
    query_vec = dense_model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scores = [float(np.dot(query_vec, kb_embeddings[i])) for i in range(len(KNOWLEDGE_BASE))]
    scored = [(KNOWLEDGE_BASE[i]["id"], scores[i]) for i in range(len(scores))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [(rank + 1, doc_id, score) for rank, (doc_id, score) in enumerate(scored[:top_k])]


def print_ranked_list(label: str, ranked: list) -> None:
    print(f"  {label}:")
    if not ranked:
        print("    (empty — no documents scored above zero)")
        return
    for rank, doc_id, score in ranked:
        doc = next(d for d in KNOWLEDGE_BASE if d["id"] == doc_id)
        print(f"    Rank {rank} | Score {score:>7.4f} | Doc {doc_id} [{doc['doc_type'].upper():<7}] {doc['text'][:50]}...")


# -----------------------------------------------------------------------
# Demo: three query types — English exact, English paraphrase, Hinglish
# -----------------------------------------------------------------------
TEST_QUERIES = [
    ("English exact match",      "premature withdrawal penalty FD"),
    ("English paraphrase",       "early closure charges fixed deposit"),
    ("Hinglish (vocabulary gap)","FD band karna hai penalty kitni hogi"),
]

for label, query in TEST_QUERIES:
    print("=" * 70)
    print(f"Query [{label}]: '{query}'")
    print("=" * 70)

    bm25_list  = bm25_ranked_list(query)
    dense_list = dense_ranked_list(query)

    print_ranked_list("BM25 ranked list", bm25_list)
    print_ranked_list("Dense ranked list", dense_list)

    # Show the score-scale mismatch explicitly
    if bm25_list and dense_list:
        bm25_top_score  = bm25_list[0][2]
        dense_top_score = dense_list[0][2]
        print(f"\n  Score scales: BM25 top score = {bm25_top_score:.4f}, Dense top score = {dense_top_score:.4f}")
        print(f"  These are NOT comparable — averaging them would be dominated by whichever is numerically larger.")
        print(f"  This is exactly why RRF uses RANK POSITION, not raw score. See Module 3.")
    print()

print("Module 2 complete. Run Module 3.")


Query [English exact match]: 'premature withdrawal penalty FD'
  BM25 ranked list:
    Rank 1 | Score  1.8869 | Doc 0 [FAQ    ] Premature withdrawal of FD incurs a 1 percent pena...
    Rank 2 | Score  1.0691 | Doc 2 [SOP    ] This SOP covers Fixed Deposit premature withdrawal...
    Rank 3 | Score  0.6129 | Doc 1 [POLICY ] Fixed Deposit premature closure is allowed subject...
  Dense ranked list:
    Rank 1 | Score  0.7929 | Doc 0 [FAQ    ] Premature withdrawal of FD incurs a 1 percent pena...
    Rank 2 | Score  0.7322 | Doc 2 [SOP    ] This SOP covers Fixed Deposit premature withdrawal...
    Rank 3 | Score  0.6006 | Doc 1 [POLICY ] Fixed Deposit premature closure is allowed subject...
    Rank 4 | Score  0.5019 | Doc 5 [POLICY ] Early exit from your deposit account will attract ...
    Rank 5 | Score  0.3912 | Doc 3 [PRODUCT] The Fixed Deposit interest rate for 24 months is 7...
    Rank 6 | Score  0.3271 | Doc 4 [PRODUCT] Senior citizens receive an additional 0.5 percent ...

  Sc

## Module 3: Reciprocal Rank Fusion — Implemented From Scratch

The core algorithm of this topic.

```text
RRF_score(d) = sum over each retriever r of:  1 / (k + rank_r(d))
```

Uses the union of both ranked lists — a document present in only one list
still contributes and can still win, which is the entire point of hybrid search
for a vocabulary-mismatch-heavy corpus.

**Requires:** Module 1, Module 2


In [3]:
"""
MODULE 3: Reciprocal Rank Fusion (RRF) — From Scratch

RRF_score(d) = sum over each retriever r of:  1 / (k + rank_r(d))

Key implementation detail: UNION not intersection.
A document missing from one list simply does not get that list's contribution --
it is NOT excluded from the final result. This is what lets dense retrieval
"rescue" documents that BM25 scored zero due to vocabulary mismatch, and lets
BM25 "rescue" exact-match documents dense retrieval ranked low.
"""

def reciprocal_rank_fusion(ranked_lists: dict, k: int = 60) -> list:
    """
    ranked_lists: dict of {retriever_name: [(rank, doc_id, score), ...]}
    k:            RRF constant (default 60, from Cormack et al. 2009)

    Returns: [(doc_id, rrf_score, contributions), ...] sorted by rrf_score descending
             contributions is a dict showing which retriever(s) contributed and at what rank
    """
    rrf_scores = {}       # doc_id -> cumulative RRF score
    contributions = {}    # doc_id -> {retriever_name: rank}

    for retriever_name, ranked_list in ranked_lists.items():
        for rank, doc_id, _score in ranked_list:
            contribution = 1.0 / (k + rank)
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + contribution
            contributions.setdefault(doc_id, {})[retriever_name] = rank

    # Sort by fused RRF score, descending
    fused = [(doc_id, rrf_scores[doc_id], contributions[doc_id]) for doc_id in rrf_scores]
    fused.sort(key=lambda x: x[1], reverse=True)
    return fused


def print_fused_results(query: str, fused: list) -> None:
    print(f"  Fused (hybrid) results for: '{query}'")
    for final_rank, (doc_id, rrf_score, contribs) in enumerate(fused, start=1):
        doc = next(d for d in KNOWLEDGE_BASE if d["id"] == doc_id)
        contrib_str = ", ".join(f"{name}=rank{rank}" for name, rank in contribs.items())
        print(f"    Final Rank {final_rank} | RRF {rrf_score:.4f} | [{contrib_str:<28}] Doc {doc_id}: {doc['text'][:45]}...")


# -----------------------------------------------------------------------
# Demo: fuse BM25 + Dense for the same three queries from Module 2
# -----------------------------------------------------------------------
for label, query in TEST_QUERIES:
    print("=" * 70)
    print(f"Query [{label}]: '{query}'")
    print("=" * 70)

    bm25_list  = bm25_ranked_list(query)
    dense_list = dense_ranked_list(query)

    fused = reciprocal_rank_fusion(
        {"BM25": bm25_list, "Dense": dense_list},
        k=60
    )

    print_fused_results(query, fused)

    # Highlight rescue cases: documents present in only ONE list
    only_dense = [doc_id for doc_id, _, c in fused if "BM25" not in c and "Dense" in c]
    only_bm25  = [doc_id for doc_id, _, c in fused if "Dense" not in c and "BM25" in c]
    if only_dense:
        print(f"\n  Rescued by Dense only (BM25 missed these): doc IDs {only_dense}")
    if only_bm25:
        print(f"  Rescued by BM25 only (Dense ranked these outside top-k): doc IDs {only_bm25}")
    print()

print("Module 3 complete. Run Module 4.")


Query [English exact match]: 'premature withdrawal penalty FD'
  Fused (hybrid) results for: 'premature withdrawal penalty FD'
    Final Rank 1 | RRF 0.0328 | [BM25=rank1, Dense=rank1     ] Doc 0: Premature withdrawal of FD incurs a 1 percent...
    Final Rank 2 | RRF 0.0323 | [BM25=rank2, Dense=rank2     ] Doc 2: This SOP covers Fixed Deposit premature withd...
    Final Rank 3 | RRF 0.0317 | [BM25=rank3, Dense=rank3     ] Doc 1: Fixed Deposit premature closure is allowed su...
    Final Rank 4 | RRF 0.0156 | [Dense=rank4                 ] Doc 5: Early exit from your deposit account will att...
    Final Rank 5 | RRF 0.0154 | [Dense=rank5                 ] Doc 3: The Fixed Deposit interest rate for 24 months...
    Final Rank 6 | RRF 0.0152 | [Dense=rank6                 ] Doc 4: Senior citizens receive an additional 0.5 per...

  Rescued by Dense only (BM25 missed these): doc IDs [5, 3, 4]

Query [English paraphrase]: 'early closure charges fixed deposit'
  Fused (hybrid) results for

## Module 4: Hybrid vs BM25-Only vs Dense-Only — Head-to-Head

Compares the top-1 result from all three strategies across a broader set of
test queries, including the exact Hinglish/English mismatch cases from
Topics 1 and 2.

**Requires:** Module 1, Module 2, Module 3


In [4]:
"""
MODULE 4: Hybrid vs BM25-only vs Dense-only — Head-to-Head Comparison

Runs a wider set of test queries and shows, for each, which strategy
got the top-1 result "right" (by inspection against the intended match).
"""

COMPARISON_QUERIES = [
    ("English exact",        "premature withdrawal penalty FD",           "Doc 0 (FAQ) or Doc 1 (policy) expected"),
    ("English paraphrase",   "early closure charges fixed deposit",        "Doc 0/1 expected via semantic match"),
    ("Hinglish premature",   "FD band karna hai penalty kitni hogi",       "Doc 0/1 expected via dense rescue"),
    ("Hinglish maturity",    "machurity ka paisa kab milega",              "No maturity doc in KB -- tests graceful handling"),
    ("Senior citizen rate",  "senior citizen extra interest rate",         "Doc 4 expected -- exact term match"),
    ("Vocabulary mismatch",  "exit my deposit early what fee",             "Doc 5 expected via BM25 partial + dense"),
]

print(f"{'Query type':<20} | {'BM25 top-1':<10} | {'Dense top-1':<12} | {'Hybrid top-1':<13} | Agreement")
print("-" * 80)

agreement_count = 0
for label, query, expected_note in COMPARISON_QUERIES:
    bm25_list  = bm25_ranked_list(query)
    dense_list = dense_ranked_list(query)
    fused      = reciprocal_rank_fusion({"BM25": bm25_list, "Dense": dense_list}, k=60)

    bm25_top1   = bm25_list[0][1] if bm25_list else None
    dense_top1  = dense_list[0][1] if dense_list else None
    hybrid_top1 = fused[0][0] if fused else None

    all_agree = (bm25_top1 == dense_top1 == hybrid_top1) and bm25_top1 is not None
    if all_agree:
        agreement_count += 1

    bm25_str   = str(bm25_top1) if bm25_top1 is not None else "NONE (0 score)"
    dense_str  = str(dense_top1)
    hybrid_str = str(hybrid_top1)
    agree_str  = "ALL AGREE" if all_agree else "DIFFER"

    print(f"{label:<20} | {bm25_str:<10} | {dense_str:<12} | {hybrid_str:<13} | {agree_str}")

print("-" * 80)
print(f"\nAll three methods agreed on {agreement_count}/{len(COMPARISON_QUERIES)} queries.")
print("Disagreement is where hybrid search earns its value -- each retriever")
print("contributes exactly where the other is weak.")

# -----------------------------------------------------------------------
# Detailed view of the most interesting disagreement case
# -----------------------------------------------------------------------
print("\n" + "=" * 70)
print("DETAILED VIEW: Hinglish query where BM25 fails, Dense partially recovers")
print("=" * 70)

query = "FD band karna hai penalty kitni hogi"
bm25_list  = bm25_ranked_list(query)
dense_list = dense_ranked_list(query)
fused      = reciprocal_rank_fusion({"BM25": bm25_list, "Dense": dense_list}, k=60)

print(f"\nQuery: '{query}'")
print(f"BM25 found {len(bm25_list)} documents with score > 0 (out of {len(KNOWLEDGE_BASE)} total)")
print(f"Dense found {len(dense_list)} documents (dense always scores everything)")
print_fused_results(query, fused)

print("\nModule 4 complete. Run Module 5.")


Query type           | BM25 top-1 | Dense top-1  | Hybrid top-1  | Agreement
--------------------------------------------------------------------------------
English exact        | 0          | 0            | 0             | ALL AGREE
English paraphrase   | 1          | 1            | 1             | ALL AGREE
Hinglish premature   | 0          | 0            | 0             | ALL AGREE
Hinglish maturity    | NONE (0 score) | 2            | 2             | DIFFER
Senior citizen rate  | 3          | 4            | 3             | DIFFER
Vocabulary mismatch  | 5          | 5            | 5             | ALL AGREE
--------------------------------------------------------------------------------

All three methods agreed on 4/6 queries.
Disagreement is where hybrid search earns its value -- each retriever
contributes exactly where the other is weak.

DETAILED VIEW: Hinglish query where BM25 fails, Dense partially recovers

Query: 'FD band karna hai penalty kitni hogi'
BM25 found 3 documents 

## Module 5: k-Value Sensitivity

Shows how changing the RRF constant `k` changes the fused ranking.
Demonstrates the "sharper vs flatter" blending behaviour described in the theory.

**Requires:** Module 1, Module 2, Module 3


In [5]:
"""
MODULE 5: k-Value Sensitivity Analysis

Tests k = 1, 10, 60 (default), 100 on the same query to show how the
constant changes how much rank position 1 is favoured over rank position 2+.
"""

K_VALUES = [1, 10, 60, 100]

query = "early closure charges fixed deposit"
bm25_list  = bm25_ranked_list(query)
dense_list = dense_ranked_list(query)

print(f"Query: '{query}'\n")
print_ranked_list("BM25 ranked list", bm25_list)
print_ranked_list("Dense ranked list", dense_list)

print(f"\n{'k value':<10} | Final ranking (doc_id sequence, best to worst)")
print("-" * 60)

for k in K_VALUES:
    fused = reciprocal_rank_fusion({"BM25": bm25_list, "Dense": dense_list}, k=k)
    ranking_str = " > ".join(str(doc_id) for doc_id, _, _ in fused)
    print(f"{k:<10} | {ranking_str}")

print("\nObservation:")
print("  Low k (1): rank-1 results dominate heavily -- a document ranked 1st")
print("             by even ONE retriever can leapfrog documents ranked 2nd by both.")
print("  High k (100): rankings flatten -- the difference between rank 1 and")
print("             rank 3 becomes small, so consistent-but-moderate rankings")
print("             across both retrievers can compete with a single strong rank-1.")
print("  Default k=60: balances these two behaviours, validated empirically")
print("             across many IR benchmarks in the original RRF paper.")

# -----------------------------------------------------------------------
# Show the actual RRF score gap between rank 1 and rank 2 at each k
# -----------------------------------------------------------------------
print("\n" + "=" * 60)
print("WHY k CONTROLS 'SHARPNESS': THE MATH")
print("=" * 60)
print(f"\n{'k':<6} | {'1/(k+1) [rank1]':<18} | {'1/(k+2) [rank2]':<18} | {'Ratio (rank1/rank2)':<20}")
print("-" * 70)
for k in K_VALUES:
    r1 = 1.0 / (k + 1)
    r2 = 1.0 / (k + 2)
    ratio = r1 / r2
    print(f"{k:<6} | {r1:<18.4f} | {r2:<18.4f} | {ratio:<20.3f}")

print("\nAt k=1: rank 1 contributes 1.33x more than rank 2 -- sharp preference.")
print("At k=100: rank 1 contributes only 1.01x more than rank 2 -- nearly flat.")
print("This confirms: lower k = sharper top-rank preference, higher k = flatter blending.")

print("\n" + "=" * 70)
print("CHAPTER 7 TOPIC 4 SUMMARY")
print("=" * 70)
print("""
Hybrid search = BM25 (exact match) + Dense (semantic match), fused via RRF.

RRF formula: RRF_score(d) = sum of 1/(k + rank_r(d)) across all retrievers r
Default k = 60 (from Cormack et al. 2009 -- empirically validated).

Why RRF over score averaging:
  BM25 scores and cosine similarity scores are on incompatible numeric scales.
  RRF uses only RANK POSITION, which is directly comparable across retrievers.

Why hybrid matters for this project:
  BM25 alone: fails on 64.4% Hinglish emails (vocabulary mismatch -> score 0)
  Dense alone: only 0.0393 discrimination gap for short emails (Topic 2 finding)
  Hybrid: each retriever rescues the other's blind spot

Implementation rule: always UNION the two ranked lists, never intersect.
  Intersection silently discards exactly the vocabulary-mismatch-rescue
  documents that make hybrid search worth doing in the first place.

Next: Topic 5 -- classify_by_keyword_rules() reframed as hand-rolled BM25
      Topic 9 -- Evaluation harness (Recall@K, MRR, NDCG@K) to properly
                 measure hybrid vs BM25-only vs dense-only on your real data
""")


Query: 'early closure charges fixed deposit'

  BM25 ranked list:
    Rank 1 | Score  3.2561 | Doc 1 [POLICY ] Fixed Deposit premature closure is allowed subject...
    Rank 2 | Score  1.7159 | Doc 5 [POLICY ] Early exit from your deposit account will attract ...
    Rank 3 | Score  0.5761 | Doc 3 [PRODUCT] The Fixed Deposit interest rate for 24 months is 7...
    Rank 4 | Score  0.5761 | Doc 4 [PRODUCT] Senior citizens receive an additional 0.5 percent ...
    Rank 5 | Score  0.3809 | Doc 2 [SOP    ] This SOP covers Fixed Deposit premature withdrawal...
  Dense ranked list:
    Rank 1 | Score  0.7343 | Doc 1 [POLICY ] Fixed Deposit premature closure is allowed subject...
    Rank 2 | Score  0.5740 | Doc 3 [PRODUCT] The Fixed Deposit interest rate for 24 months is 7...
    Rank 3 | Score  0.5375 | Doc 2 [SOP    ] This SOP covers Fixed Deposit premature withdrawal...
    Rank 4 | Score  0.5255 | Doc 0 [FAQ    ] Premature withdrawal of FD incurs a 1 percent pena...
    Rank 5 | Score  0.